# 01 · Chain-of-Thought Reasoning

> **Source notes:** `CoTReasoning.md` + `CoTReasoning_Supplement.md`

How do LLMs "think" step-by-step before answering? This notebook covers:
- Visible vs. hidden CoT reasoning
- Self-Consistency (majority-vote over N chains)
- Tree of Thoughts and other reasoning patterns
- PRM vs. ORM reward models
- Common failure modes

All demos use a **local SLM via Ollama** — no API key, no cloud cost.

**Running example throughout:** Mamma Rosa's PizzaBot — the multi-constraint query *\"What's the cheapest gluten-free pizza under 600 calories for delivery by 7 pm?\"* Each section shows how CoT decomposition handles one layer of that constraint chain. See [AIPrimer.md](../AIPrimer.md) for the full system context.

## 0 · Environment Setup

### A — Install Ollama (one-time, run in a terminal)
```bash
# Windows
winget install Ollama.Ollama
# macOS
brew install ollama
# Linux
curl -fsSL https://ollama.ai/install.sh | sh
```

### B — Pull a small local model (~2 GB, run in a terminal)
```bash
ollama pull phi3:mini       # Microsoft Phi-3 Mini, fast and fits in 4 GB RAM
# alternatives: ollama pull llama3.2   |   ollama pull mistral
```

Make sure the Ollama desktop app is open **or** run `ollama serve` in a terminal before executing cells below.

### C — Python packages

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ollama", "-q"], check=True)
print("Done. Make sure Ollama is running (`ollama serve` or the desktop app).")

## 1 · What Is Chain-of-Thought (CoT)?

An LLM predicts the next token. On multi-step problems it can "jump" straight to a wrong answer without checking intermediate logic.

**CoT prompting** inserts intermediate reasoning steps between the question and the answer:

| Variant | How It Works | Visibility |
|---------|-------------|------------|
| **Visible CoT** | Prompt says "think step by step"; steps appear in output | You see every reasoning step |
| **Hidden Reasoning Tokens** | Internal scratchpad (o1, o3, DeepSeek-R1) | You only see the final answer |

Two ways to trigger visible CoT:
- **Zero-shot:** append *"Think step by step."* to any prompt.
- **Few-shot:** provide 1–3 worked examples showing the step pattern.

The running example shows clearly that CoT produces a more grounded answer even for this simple maths problem.

In [ ]:
import ollama

MODEL = "phi3:mini"   # change to whichever model you pulled

def chat(system: str, user: str, temperature: float = 0.0) -> str:
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        options={"temperature": temperature},
    )
    return resp["message"]["content"]

QUERY = (
    "Which Mamma Rosa pizza is gluten-free and has the fewest calories?"
)

# Without CoT
direct = chat(system="Answer concisely.", user=QUERY)
print("=== Without CoT ===")
print(direct)

# With Zero-Shot CoT
cot = chat(
    system="Think step by step before giving the final answer.",
    user=QUERY,
)
print("\n=== With CoT ===")
print(cot)


## 2 · Self-Consistency — Majority Vote Over N Chains

**Problem:** A single CoT chain can still go wrong.  
**Fix:** Sample **N independent** chains (temperature > 0) and take the **majority vote**.

```
Query  →  Chain 1  →  Veggie Feast GF  ─┐
       →  Chain 2  →  Veggie Feast GF  ─┤─ majority  →  Veggie Feast GF  ✓
       →  Chain 3  →  Napoli GF        ─┤
       →  Chain 4  →  Veggie Feast GF  ─┘
```

**When to use:** high-stakes tasks — medical, financial, legal math — where 5–20× token cost is acceptable.  
**When to skip:** latency-sensitive apps; each extra chain adds cost and delay.


In [ ]:
import re
from collections import Counter

def extract_answer(text: str) -> str:
    """Pull the item name after 'Answer:' from model output."""
    m = re.search(r"Answer:\s*(.+)", text, re.IGNORECASE)
    if m:
        return m.group(1).strip().rstrip(".")
    return text.strip()[-40:]

def self_consistency(user: str, n: int = 5, temp: float = 0.7) -> str:
    answers = []
    for i in range(n):
        resp = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system",
                 "content": "Think step by step. End with 'Answer: <item_name>'."},
                {"role": "user", "content": user},
            ],
            options={"temperature": temp},
        )
        ans = extract_answer(resp["message"]["content"])
        answers.append(ans)
        print(f"  Path {i+1}: {ans}")
    majority = Counter(answers).most_common(1)[0][0]
    return majority

print("Sampling 5 reasoning paths ...")
winner = self_consistency(QUERY, n=5)
print(f"\nMajority answer: {winner}")


## 3 · Reasoning Architecture Taxonomy

| Pattern | Structure | Best For | Main Risk |
|---------|-----------|----------|-----------|
| **CoT (Linear)** | Sequential steps | Most tasks; cheap | Mid-chain hallucination |
| **Self-Consistency** | N paths + majority vote | High-stakes QA | N× token cost |
| **Tree of Thoughts (ToT)** | BFS/DFS over branches | Puzzles, open-ended search | Exponential branching |
| **Graph of Thoughts (GoT)** | DAG — merge & split paths | Research synthesis | Hard to implement |
| **Reflexion** | CoT + self-critique loop | Code generation | Added latency |
| **LATS** | Monte Carlo Tree Search | Complex planning | Very high compute |
| **ReWOO** | Separate Planner + Executor | Deterministic pipelines | Rigid; no mid-plan correction |

### Tree of Thoughts — Core Idea

Instead of one chain, maintain a **tree** of partial solutions:
1. At each node, generate K candidate "next thoughts"
2. Evaluate each branch (LLM scores 1–10)
3. Prune weak branches; expand the best
4. BFS (broad exploration) or DFS (go deep first)

In [ ]:
# Tree of Thoughts — minimal 2-level demo

def generate_thoughts(context: str, n: int = 3) -> list:
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": (
            f"Context so far: {context}\n\n"
            f"List exactly {n} different next reasoning steps, one per line, numbered."
        )}],
        options={"temperature": 0.8},
    )
    lines = [l.strip() for l in resp["message"]["content"].splitlines() if l.strip()]
    return lines[:n]

def score_thought(thought: str, goal: str) -> float:
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": (
            f"Goal: {goal}\nThought: {thought}\n\n"
            "Rate how useful this thought is (1=useless, 10=perfect). Reply with a number only."
        )}],
        options={"temperature": 0.0},
    )
    m = re.search(r"\d+", resp["message"]["content"])
    return float(m.group()) if m else 5.0

GOAL = "Identify the cheapest gluten-free Mamma Rosa pizza under 600 calories."
ROOT = "We need to: filter menu by GF allergen, check calorie counts, then sort by price."

print("=== Level-1 branches ===")
l1 = generate_thoughts(ROOT, n=3)
scored = [(score_thought(t, GOAL), t) for t in l1]
for s, t in sorted(scored, reverse=True):
    print(f"  [{s:.0f}/10] {t[:90]}")

best = max(scored, key=lambda x: x[0])[1]
print(f"\nBest L1: '{best[:80]}...'")
print("\n=== Level-2 branches from best ===")
for t in generate_thoughts(f"{ROOT} {best}", n=3):
    print(f"  - {t}")


## 4 · Hidden Reasoning Tokens & Reward Models

### Hidden Reasoning Tokens (o1, o3, DeepSeek-R1)
- Model uses an **invisible scratchpad** before producing the visible response
- Billed as completion tokens — check `usage.completion_tokens_details.reasoning_tokens`
- Adaptive depth: simple queries use ~10 tokens; hard proofs use ~4 000+

### PRM vs. ORM — How Reasoning Models Are Trained

| | Outcome Reward Model (ORM) | Process Reward Model (PRM) |
|--|---------------------------|---------------------------|
| **Signal** | Correctness of final answer | Correctness of **each step** |
| **Risk** | Right answer via wrong path | More expensive labels |
| **Best for** | General tasks | Math, multi-step reasoning |

PRM is used in o1-class training — it prevents the model from "getting lucky" with flawed intermediate steps.

### Common CoT Failure Modes

| Failure | Description | Mitigation |
|---------|-------------|-----------|
| **Unfaithful reasoning** | Visible chain is post-hoc rationalization | Require tool-verified intermediate values |
| **Sycophancy** | Chain bends toward the implied expected answer | Use neutral prompts; don't hint at expected answer |
| **Overthinking** | Model second-guesses its correct earlier steps | Cap reasoning budget |
| **Hallucinated observations** | Model fabricates tool results | Always use real tool calls |
| **Context length collapse** | Early observations forgotten as scratchpad grows | Summarise older scratchpad entries |

## 5 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| CoT | Intermediate steps → better accuracy on multi-step tasks |
| Self-Consistency | N paths + majority vote → reliability at N× cost |
| ToT | BFS/DFS over thought branches → exploration-heavy tasks |
| Hidden tokens | Model reasons invisibly; you pay in cost and latency |
| PRM | Reward each step not just the answer → sounder reasoning |
| Unfaithful CoT | Visible chain can be decorative; verify with real tool calls |

**Next:** `02-rag-embeddings/notebook.ipynb` — how agents retrieve knowledge they weren't trained on.

## 6 · PizzaBot Connection — CoT in a Real System

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The multi-constraint query `"cheapest gluten-free pizza under 600 calories for delivery by 7 pm"` is the PizzaBot version of Chain-of-Thought decomposition. The chain decomposes a multi-constraint customer request into verifiable sequential sub-queries:

```
Thought: The query has four constraints — gluten-free, under 600 kcal, deliverable, cheapest.
         Satisfy them in dependency order (filter first, price-sort last).

Step 1 — Allergen filter:   retrieve_from_rag("gluten-free pizzas")
Step 2 — Calorie filter:    retrieve_from_rag("calorie counts gluten-free options")
Step 3 — Availability:      check_item_availability(store_id, candidate_items)
Step 4 — Price sort:        retrieve_from_rag("prices Veggie Feast Margherita GF") → sort ascending
Step 5 — Answer:            FINAL_ANSWER: "Veggie Feast GF — £11.99, arrives in ~28 min"
```

**Why CoT matters here:** without step-by-step decomposition the model tries to satisfy all four constraints in one pass and likely misses one — confabulating a calorie count or skipping the allergen check. With explicit CoT, each step is verifiable and the full trace is debuggable.

**PRM relevance:** a Process Reward Model would score each individual step (was the allergen check correct? was the cheapest item actually selected?), not just the final answer. This is critical for safety: a correct final recommendation built on a wrong allergen check should score low at the step level.
